# Module 13: Data networks, AFDX and IMA
A notebook that checks the module's numbers (Tooley Ch.16, 18). Each part: a question, code, then a "Reading the result" cell.

## 1. The BAG
Table 16.1 and the largest BAG for 100 and 400 messages a second.

In [1]:
# Bang 16.1 va chon BAG
BAGS = [1, 2, 4, 8, 16, 32, 64, 128]
print({b: 1000 / b for b in BAGS})
def max_bag(rate): return max(b for b in BAGS if 1000 / b >= rate)
print('100/s ->', max_bag(100), 'ms; 400/s ->', max_bag(400), 'ms (khoa in cua sach: 4 ms)')

{1: 1000.0, 2: 500.0, 4: 250.0, 8: 125.0, 16: 62.5, 32: 31.25, 64: 15.625, 128: 7.8125}
100/s -> 8 ms; 400/s -> 2 ms (khoa in cua sach: 4 ms)


**Reading the result.** BAGs of 1 to 128 ms give 1,000 to 7.8125 Hz. 100 a second gives 8 ms (matching Tooley's example); 400 a second gives 2 ms because 4 ms is only 250 Hz.

## 2. VL bit rate
The maximum bit rate for 1,518-byte frames with a 4 ms BAG and 256-byte frames with a 128 ms BAG.

In [2]:
# Toc do bit lon nhat cua VL = L * 8 * 1000 / BAG
for L, bag in ((1518, 4), (256, 128), (64, 2)): print(L, 'byte,', bag, 'ms ->', L * 8 * 1000 / bag, 'bit/s')

1518 byte, 4 ms -> 3036000.0 bit/s
256 byte, 128 ms -> 16000.0 bit/s
64 byte, 2 ms -> 256000.0 bit/s


**Reading the result.** 3,036,000 bit/s (≈ 3 Mbit/s, Tooley's example) and 16,000 bit/s (Chapter 16 Q6).

## 3. The AFDX frame
The minimum and maximum length and the transmit time.

In [3]:
# Do dai khung AFDX tu cac truong
def frame_len(payload):
    pad = max(0, 17 - payload)
    return 6 + 6 + 2 + 20 + 8 + payload + pad + 1 + 4
print('payload 1:', frame_len(1), '| payload 17:', frame_len(17), '| payload 1471:', frame_len(1471))
for nbytes, rate in ((1518, 10e6), (1518, 100e6)): print(nbytes, 'byte o', rate / 1e6, 'Mbit/s:', nbytes * 8 / rate * 1e3, 'ms')
print('khoang cach giua khung 12 byte o 100 Mbit/s:', 12 * 8 / 100e6 * 1e6, 'us')

payload 1: 64 | payload 17: 64 | payload 1471: 1518
1518 byte o 10.0 Mbit/s: 1.2144 ms
1518 byte o 100.0 Mbit/s: 0.12144 ms
khoang cach giua khung 12 byte o 100 Mbit/s: 0.96 us


**Reading the result.** A 1-byte payload (16 of padding) and a 17-byte payload both give 64 bytes; a 1,471-byte payload gives 1,518 bytes. The longest frame takes 1.2144 ms at 10 Mbit/s and 0.12144 ms at 100 Mbit/s; the 12-byte gap is 0.96 μs at 100 Mbit/s.

## 4. Sequence number and exponential back-off
What follows SN 255, and the back-off interval after 1 to 5 collisions?

In [4]:
# So thu tu quay vong va lui ham mu nhi phan
sn, seq = 253, []
for _ in range(5): seq.append(sn); sn = 1 if sn == 255 else sn + 1
print(seq)
print([2 ** (n - 1) for n in range(1, 6)], '(boi cua T sau lan va cham thu n)')

[253, 254, 255, 1, 2]
[1, 2, 4, 8, 16] (boi cua T sau lan va cham thu n)


**Reading the result.** After 255 comes 1 (not 0). The back-off intervals are 1T, 2T, 4T, 8T, 16T.